#  Student performance Tracker model


## Step 1. Business understanding
Educational institutions want to intervene early with students who are at risk of underperforming, but they currently have no data-driven way to identify these students before final scores are recorded. The goal of this project is to build a predictive model that estimates a student's academic performance based on demographic and preparatory factors, so that schools, parents, and students can identify risk early and put targeted support measures (tutoring, test-prep enrollment, resource allocation) in place before performance suffers.
## Solution
We will develop a binary classification model that predicts whether a student is likely to Pass or Fail based on their  attributes (gender, race/ethnicity, parental level of education, lunch type and test preparation course completion). This model will serve as an early-warning system, allowing educators and parents to flag at-risk students before final assessments and put support measures in place.

As a secondary feature, once a student is flagged, the user can choose to run a regression model that predicts their actual expected scores (math, reading, writing) — giving a more granular view of how much intervention might be needed, not just a flag.

## Step 2 Data Preparation

## 2.1 Importing the Libraries

In [3]:
# import the necessary libraries
import pandas as pd
import matplotlib.pyplot as pyplot
import seaborn as sns
from sklearn.preprocessing import LabelEncoder


## 2.2 Loading The Dataset

In [4]:
# load the dataset
df=pd.read_csv("StudentsPerformance.csv")
df.head()


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


## 2.3 Data Inspection

In [5]:
# Check and understand the data
df.shape

(1000, 8)

The dataset has 1000 rows and 8 columns

In [6]:
df.describe()

,math score,reading score,writing score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000


from the analysis the percentage that shows the student have passed is 80% because for all the 3 subjects, that is the 3rd quartile. we will use that to group students whether they have passed or failed

In [7]:
df.describe(include='all').T

for c in ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']:
    print(c, df[c].unique())

gender <ArrowStringArray>
['female', 'male']
Length: 2, dtype: str
race/ethnicity <ArrowStringArray>
['group B', 'group C', 'group A', 'group D', 'group E']
Length: 5, dtype: str
parental level of education <ArrowStringArray>
[ 'bachelor's degree',       'some college',    'master's degree',
 'associate's degree',        'high school',   'some high school']
Length: 6, dtype: str
lunch <ArrowStringArray>
['standard', 'free/reduced']
Length: 2, dtype: str
test preparation course <ArrowStringArray>
['none', 'completed']
Length: 2, dtype: str


## 2.4 Data Cleaning

## 2.4.1 Handling duplicates

In [8]:
# check for duplicates
full_dupes = df.duplicated().sum()
print(f"fully duplicated rows: {full_dupes}")



fully duplicated rows: 0


In [9]:
# Drop duplicates
df = df.drop_duplicates()


## 2.4.2 Handle Missing values

In [10]:
 
# check for null values
df.isnull().sum()


gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [11]:
# Handle null Values
print("Missing values per column:")
print(df.isnull().sum())

placeholder_values = ['NA', 'N/A', 'na', 'n/a', 'Unknown', 'UNKNOWN', '?', '-', '', 'None', 'null']
print("\nRows with placeholder text per column:")
for c in df.select_dtypes(include='object').columns:
    hits = df[c].isin(placeholder_values).sum()
    if hits:
        print(f"  {c}: {hits}")

Missing values per column:
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

Rows with placeholder text per column:


C:\Users\Administrator\AppData\Local\Temp\ipykernel_8632\879575525.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include='object').columns:


In [12]:
# check for general information of the data
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race/ethnicity               1000 non-null   str  
 2   parental level of education  1000 non-null   str  
 3   lunch                        1000 non-null   str  
 4   test preparation course      1000 non-null   str  
 5   math score                   1000 non-null   int64
 6   reading score                1000 non-null   int64
 7   writing score                1000 non-null   int64
dtypes: int64(3), str(5)
memory usage: 103.5 KB


## 2.4.3 Handle missing values for categorical and numerical columns

In [13]:
# Handle missing values
numeric_cols = ['math score', 'reading score', 'writing score']
for c in numeric_cols:
    if df[c].isnull().any():
        df[c] = df[c].fillna(df[c].median())

cat_cols = ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']
for c in cat_cols:
    if df[c].isnull().any():
        df[c] = df[c].fillna(df[c].mode()[0])

## 2.4.4 Outlier Detection and removal of errors

In [14]:
# outlier detection using IQR 
outlier_columns =['math score', 'reading score', 'writing score']

def count_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()
for c in outlier_columns:
    print(f"{c}: {count_outliers(df[c])} outliers")

math score: 8 outliers
reading score: 6 outliers
writing score: 5 outliers


In [15]:
# quick range check
for c in outlier_columns:
    print(f"{c}: min={df[c].min()}, max={df[c].max()}")

math score: min=0, max=100
reading score: min=17, max=100
writing score: min=10, max=100


In [16]:
# Check using boxplots
fig, axes = pyplot.subplots(1, 3, figsize=(20, 4))
for ax , c in zip(axes, numeric_cols):
    ax.boxplot(df[c])
    ax.set_title(c)
pyplot.tight_layout()
pyplot.show()    

C:\Users\Administrator\AppData\Local\Temp\ipykernel_8632\4051268751.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pyplot.show()


## From the data quality check above this is what was discovered:
#### The dataset consists of 1000 rows and 8 columns
#### There is no null entries 
#### There is no duplicated values
#### We have 3 columns holding numerical values(math score, reading score, writing score)
#### We have few outliers from the three target variables that means we can let them be as marks can be higher than the rest.
#### We also introduced a column, top_performer with values pass or fail. it was deduced from the scores of the three available subjects such that if a student scores 80% average from the three then the student has passed else failed 

## Step 3 Exploratory data analysis
In this step we are exploring the cleaned data set to understand the distribution of variables ,identify patterns  and relationships, and examine factors that may be associated with student performance.

## 3.0 Numeric Analysis- Univariate Analysis

In [17]:
fig, axes = pyplot.subplots(1, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, c in zip(axes, numeric_cols):
    ax.hist(df[c], bins=20, color='#5B8FF9', edgecolor='white')
    ax.set_title(f"Distribution of {c}")

pyplot.tight_layout()
pyplot.show()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_8632\1879155219.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pyplot.show()


## 3.1 Feature Engineering

In [18]:
# Get the average of the 3 scores
df['average_score'] = df[['math score', 'reading score', 'writing score']].mean(axis=1)
# Derive the performance label: pass if average >= 80, else fail
df['top_performer'] = df['average_score'].apply(lambda x: 'pass' if x >= 80 else 'fail')
# show the dataset
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score,average_score,top_performer
0,female,group B,bachelor's degree,standard,none,72,72,74,72.666667,fail
1,female,group C,some college,standard,completed,69,90,88,82.333333,pass
2,female,group B,master's degree,standard,none,90,95,93,92.666667,pass
3,male,group A,associate's degree,free/reduced,none,47,57,44,49.333333,fail
4,male,group C,some college,standard,none,76,78,75,76.333333,fail


## 3.2 Categorical Variables- Univariate Analysis

In [19]:
fig, axes = pyplot.subplots(2, 3, figsize=(16, 8))
# Convert the 2D axes array into a 1D array
axes = axes.flatten()
for ax, c in zip(axes, cat_cols):
    df[c].value_counts().sort_index().plot(
        kind='bar',
        ax=ax,
        color='#5B8FF9'
    )
    
    ax.set_title(c)
    ax.set_xlabel(c)
    ax.set_ylabel('Count')

pyplot.tight_layout()

pyplot.show()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_8632\325596937.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pyplot.show()


## 3.3 Target Variable analysis

In [20]:
# top performer
counts = df['top_performer'].value_counts()
percentages = df['top_performer'].value_counts(normalize=True) * 100

print(counts)
print(percentages.round(2))

top_performer
fail    802
pass    198
Name: count, dtype: int64
top_performer
fail    80.2
pass    19.8
Name: proportion, dtype: float64


In [21]:
# visual representation
fig, ax = pyplot.subplots(figsize=(5, 4))
counts.plot(kind='bar', color=['#5B8FF9', '#F76965'], ax=ax)
ax.set_xticklabels(['fail (0)', 'pass (1)'], rotation=0)
ax.set_title('top performers distribution')
ax.set_ylabel('Number of students')
pyplot.tight_layout()
pyplot.show()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_8632\146136917.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pyplot.show()


## 3.4 Multivariate Analysis
We will be comparing all numeric variables at a time using correlation matrix.

In [22]:

# Calculate the correlation matrix for the numerical features
numeric_cols = [col for col in df.select_dtypes(include=['number']).columns ]
correlation_matrix = df[numeric_cols].corr()
# Visualize the new correlation matrix using a heatmap
pyplot.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
pyplot.title('Correlation Matrix Heatmap (numerical variables)')
pyplot.show()

# Print the correlation values
print(correlation_matrix)

               math score  reading score  writing score  average_score
math score       1.000000       0.817580       0.802642       0.918746
reading score    0.817580       1.000000       0.954598       0.970331
writing score    0.802642       0.954598       1.000000       0.965667
average_score    0.918746       0.970331       0.965667       1.000000


C:\Users\Administrator\AppData\Local\Temp\ipykernel_8632\44645659.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pyplot.show()


From the correlation matrix, the numerical features are strongly correlated, positively.

## 3.5 Pairplot


In [23]:
# pairplot analysis
pairplot = sns.pairplot(df, x_vars=['math score', 'writing score', 'reading score'], y_vars=['math score','writing score', 'reading score'], height=8, aspect=1, )
pyplot.suptitle('Numerical features score pairplot', y=1.02, fontsize= 22)
# Increase subplots axes fontsice
for ax in pairplot.axes.flatten():
    ax.set_xlabel(ax.get_xlabel(), fontsize=18)
    ax.set_ylabel(ax.get_ylabel(), fontsize=18)

### the scores are positively correlated as one increases the next  increases too.

## 3.6 Bivariate Analysis

In [24]:
# Gender distribution
sns.countplot(x= 'gender', hue='top_performer', data=df)

<Axes: xlabel='reading score', ylabel='Count'>

In [25]:
# Check the distribution of top performers per race/ethnicity
sns.countplot(x='race/ethnicity',hue='top_performer', data=df)

<Axes: xlabel='reading score', ylabel='Count'>

In [26]:
# Check the distribution of top performers using parents education level
sns.countplot(x='parental level of education', hue='top_performer', data=df) 
pyplot.xticks(rotation=45, ha="right")

([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
 [Text(0, 0, 'female'),
  Text(1, 0, 'male'),
  Text(2, 0, 'group B'),
  Text(3, 0, 'group C'),
  Text(4, 0, 'group A'),
  Text(5, 0, 'group D'),
  Text(6, 0, 'group E'),
  Text(7, 0, "bachelor's degree"),
  Text(8, 0, 'some college'),
  Text(9, 0, "master's degree"),
  Text(10, 0, "associate's degree"),
  Text(11, 0, 'high school'),
  Text(12, 0, 'some high school')])

In [27]:
# check if top performing is associated with lunch
sns.countplot(x='lunch', hue='top_performer', data = df)

<Axes: xlabel='reading score', ylabel='Count'>

In [28]:
# Check the differences in performance for students in regards to test preparation cours
sns.countplot(x='test preparation course', hue='top_performer', data= df)

<Axes: xlabel='reading score', ylabel='Count'>

## 4 Modelling

## 4.1 import the necessary libraries


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import sklearn.metrics as metrics

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
# evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score
)
from sklearn.tree import plot_tree



## 4.2 Data Pre Processing

In [38]:
# Create the text encoder preprocessor
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                  ],
    remainder="drop",
)



## 4.3 Data Splitting

In [31]:
X =df[cat_cols]
y = df['top_performer']
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, random_state= 42, stratify=y)

## Random Forest Model

In [44]:
# Bundle the encoder and model into a single Pipeline workflow
rf_pipeline = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", RandomForestClassifier(random_state=42, class_weight='balanced'))]
)

## 4.4 Fitting 


In [45]:
# Train the entire pipeline
rf_pipeline.fit(X_train, y_train)

# Predictions
y_pred = rf_pipeline.predict(X_test)


## 4.5 Model Evaluation

In [47]:
# Model Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:")
print(metrics.classification_report(y_test, y_pred))

Accuracy: 0.66
Classification Report:
              precision    recall  f1-score   support

        fail       0.84      0.71      0.77       160
        pass       0.28      0.45      0.35        40

    accuracy                           0.66       200
   macro avg       0.56      0.58      0.56       200
weighted avg       0.73      0.66      0.69       200



#### With only demographic and preparatory factors (gender, race/ethnicity, parental education, lunch, test prep), the model can identify 45% of genuine top performers, but at the cost of a high false-positive rate (many students flagged as potential top performers won't actually reach the 80% bar — precision of only 0.28 on "pass"). This tells us these five factors carry a real but limited signal — they're not powerful enough alone to reliably predict who will excel, but they do meaningfully separate risk levels better than chance (56% macro F1 vs. an uninformed baseline). Practically: schools should treat the model's "predicted pass" flag as a signal to investigate further, not a certainty — e.g., a flagged student might warrant an actual diagnostic assessment, not just an assumption they'll succeed unaided. The weak precision here also suggests that structural/preparatory factors alone are insufficient — richer features (attendance, prior grades, study hours) would likely be needed for a more decisive tool.

In [68]:
# confusion matrix
cm_display = ConfusionMatrixDisplay.from_estimator(rf_pipeline, X_test, y_test, cmap=pyplot.cm.Blues )
pyplot.title("Confusion Matrix for Random Forest Model")
pyplot.show()


C:\Users\Administrator\AppData\Local\Temp\ipykernel_8632\4065196991.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pyplot.show()


In [51]:
# Sensitivity for Random Forest
rf_sensitivity = recall_score(y_test, y_pred, pos_label='pass')
print("Random Forest Sensitivity:", rf_sensitivity)
   

Random Forest Sensitivity: 0.45


## Model 2- Logistic Regression


In [64]:
# Logistic Regression Model
log_pipeline = Pipeline(steps= [("preprocessor", preprocessor), ("classifier", LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))])
# Train The model
log_pipeline.fit(X_train, y_train)
# Prediction
log_predictions = log_pipeline.predict(X_test)


In [65]:
# Model Evaluation
accuracy = accuracy_score(y_test, log_predictions)
print("Logistic Regression Model Evaluation:")
print("Accuracy:", accuracy)


Logistic Regression Model Evaluation:
Accuracy: 0.7


In [67]:
# classificatio report
classification_report = metrics.classification_report(y_test, log_predictions)
print(classification_report)

              precision    recall  f1-score   support

        fail       0.89      0.71      0.79       160
        pass       0.36      0.65      0.46        40

    accuracy                           0.70       200
   macro avg       0.63      0.68      0.63       200
weighted avg       0.78      0.70      0.73       200



#### Logistic Regression looks better on the surface (0.795 vs 0.66), but its recall on "pass" is only 0.05 — it's catching just 2 out of 40 actual top performers in your test set. It's basically defaulting to predicting "fail" almost every time and getting rewarded for it, because 80% of students are "fail" anyway. Random Forest, despite lower overall accuracy, is doing the job you actually need — catching 45% of true top performers.

## 4.6 Model Selection

Two models : Random Forest and Logistic Regression — were each tested with and without class weighting to address the 80/20 class imbalance. Logistic Regression, once weighted, outperformed Random Forest on the metric aligned with the business objective: identifying 65% of genuine top performers (recall) with the best F1-score (0.46) among all four model/weighting combinations tested. This shows that model choice and imbalance handling interact — a technique that helps one algorithm significantly (Logistic Regression) may help another only modestly (Random Forest). Logistic Regression (balanced) was therefore selected as the final model, prioritizing the minority-class recall and F1 relevant to early-warning use cases over raw accuracy.

## 4.7 Confirmation it is not overfitting

In [69]:
print("Train accuracy:", log_pipeline.score(X_train, y_train))
print("Test accuracy:", log_pipeline.score(X_test, y_test))

Train accuracy: 0.66125
Test accuracy: 0.7


## 4.8 Model Coefficients


In [70]:
coefs = log_pipeline.named_steps['classifier'].coef_[0]
feature_names = log_pipeline.named_steps['preprocessor'].get_feature_names_out()
coef_df = pd.Series(coefs, index=feature_names).sort_values(ascending=False)
print(coef_df)

cat__parental level of education_master's degree       0.607940
cat__test preparation course_completed                 0.530016
cat__lunch_standard                                    0.432787
cat__race/ethnicity_group E                            0.373970
cat__parental level of education_bachelor's degree     0.296815
cat__gender_female                                     0.228284
cat__race/ethnicity_group D                            0.225874
cat__parental level of education_associate's degree    0.214965
cat__race/ethnicity_group B                            0.059180
cat__parental level of education_some college         -0.081443
cat__race/ethnicity_group C                           -0.213603
cat__gender_male                                      -0.274110
cat__lunch_free/reduced                               -0.478613
cat__parental level of education_some high school     -0.480826
cat__race/ethnicity_group A                           -0.491247
cat__test preparation course_none       

### Insights and Reccommendations
Insights
Of the five demographic/preparatory features tested, parental education level and test preparation course completion are the strongest predictors of a student reaching the 80% "top performer" threshold, followed closely by lunch type (a socioeconomic proxy).
Students with parents holding a master's degree or who completed test prep show the clearest positive shift toward passing; students with high-school-only parental education, no test prep, or free/reduced lunch show the clearest negative shift.
Race/ethnicity also shows a meaningful performance gap across groups: a real pattern in the data, but one to monitor for equity purposes rather than act on directly.
These features carry real but limited signal: the best model (Logistic Regression, balanced) reaches 65% recall and 0.46 F1 on identifying top performers, meaning demographic/preparatory data alone can meaningfully flag risk but isn't sufficient for high-precision prediction. Academic history or behavioral data would likely improve this.
Recommendations
Expand test-prep access and encouragement, prioritizing students on free/reduced lunch and lower parental-education households: this is the most actionable lever available, since it's the one factor schools can directly influence.
Use lunch status and parental education as early risk indicators to proactively offer tutoring or academic support, rather than waiting for scores to reveal underperformance.
Treat race/ethnicity findings as an equity-monitoring signal not a targeting criterion: investigate underlying access/resource gaps for lower-performing groups rather than building interventions around demographic group membership.
Use the model as a screening tool, not a final decision-maker : given its moderate precision, a "predicted pass/fail" flag should prompt further assessment of a student, not a standalone judgment.

## 5.0 Model Deployment

## 5.1 Save the logistic regression trained pipeline

In [71]:
# Save the logistic regression model
joblib.dump(log_pipeline, 'student_performance_pipeline.pkl')
print("Model saved successfully!")

Model saved successfully!


## 5.2 Basic App structure

In [73]:
import streamlit as st
import pandas as pd
import joblib

# Load the trained pipeline
model = joblib.load('student_performance_pipeline.pkl')

st.title("Student Performance Predictor")
st.write("Predict whether a student is likely to be a top performer based on demographic and preparatory factors.")

# Input widgets — one per feature your model was trained on
gender = st.selectbox("Gender", ["female", "male"])
race = st.selectbox("Race/Ethnicity", ["group A", "group B", "group C", "group D", "group E"])
parent_edu = st.selectbox("Parental Level of Education", 
                           ["some high school", "high school", "some college", 
                            "associate's degree", "bachelor's degree", "master's degree"])
lunch = st.selectbox("Lunch Type", ["standard", "free/reduced"])
test_prep = st.selectbox("Test Preparation Course", ["none", "completed"])

if st.button("Predict"):
    input_df = pd.DataFrame({
        'gender': [gender],
        'race/ethnicity': [race],
        'parental level of education': [parent_edu],
        'lunch': [lunch],
        'test preparation course': [test_prep]
    })
    
    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0]
    
    if prediction == 'pass':
        st.success(f"Predicted: Top Performer  (confidence: {probability[list(model.classes_).index('pass')]:.1%})")
    else:
        st.warning(f"Predicted: At Risk (confidence: {probability[list(model.classes_).index('fail')]:.1%})")
        

2026-09-20 09:35:44.705 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-09-20 09:35:45.182 
  command:

    streamlit run c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-20 09:35:45.182 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-20 09:35:45.185 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-20 09:35:45.187 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-20 09:35:45.189 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-20 09:35:45.190 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-20 09:35:45.192 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-20 09:35:45.194 Thread 'MainThread': missing ScriptRunContext! Thi